In [0]:
# Check the catalog and schema currently selected.
display(spark.sql("""
    SELECT
        current_catalog() AS current_catalog,
        current_schema() AS current_schema
"""))

current_catalog,current_schema
workspace,default


In [0]:
 #List the catalogs available to your account.
display(spark.sql("SHOW CATALOGS"))

catalog
samples
system
workspace


In [0]:
# Create the schema for the project tables.
spark.sql("""CREATE SCHEMA IF NOT EXISTS workspace.mortgage_risk""")

# Use the volume where your raw files are stored.
spark.sql("""CREATE VOLUME IF NOT EXISTS workspace.default.freddie_mac_raw""")

# Check that the raw-data volume exists.
display(spark.sql("""SHOW VOLUMES IN workspace.default"""))

database,volume_name
default,freddie_mac_raw
default,test


In [0]:
#Set the folder paths
raw_path = "/Volumes/workspace/default/freddie_mac_raw/"
project_path = raw_path + "DB Project/"

dbutils.fs.mkdirs(project_path)

display(dbutils.fs.ls(project_path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2016/,sample_2016/,0,1792183223333
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2017/,sample_2017/,0,1792183223333
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2018/,sample_2018/,0,1792183223333
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2019/,sample_2019/,0,1792183223333
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2020/,sample_2020/,0,1792183223333
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2021/,sample_2021/,0,1792183223334
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2022/,sample_2022/,0,1792183223334
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2023/,sample_2023/,0,1792183223334
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2024/,sample_2024/,0,1792183223334
dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2025/,sample_2025/,0,1792183223334


In [0]:
project_path = "/Volumes/workspace/default/freddie_mac_raw/DB Project/"

# Find all text files, including those inside year subfolders.
def find_text_files(folder):
    results = []

    for item in dbutils.fs.ls(folder):
        if item.isDir():
            results.extend(find_text_files(item.path))
        elif item.name.endswith(".txt"):
            results.append(item)

    return results

project_files = find_text_files(project_path)

# Display filenames, sizes, and complete paths.
file_inventory = [
    (f.name, round(f.size / (1024 ** 2), 2), f.path)
    for f in project_files
]

display(
    spark.createDataFrame(
        file_inventory,
        "filename STRING, size_mb DOUBLE, path STRING"
    ).orderBy("filename")
)

print(f"Total text files found: {len(project_files)}")


filename,size_mb,path
sample_orig_2016.txt,5.98,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2016/sample_orig_2016.txt
sample_orig_2017.txt,6.0,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2017/sample_orig_2017.txt
sample_orig_2018.txt,6.05,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2018/sample_orig_2018.txt
sample_orig_2019.txt,6.08,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2019/sample_orig_2019.txt
sample_orig_2020.txt,5.94,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2020/sample_orig_2020.txt
sample_orig_2021.txt,6.0,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2021/sample_orig_2021.txt
sample_orig_2022.txt,6.05,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2022/sample_orig_2022.txt
sample_orig_2023.txt,6.05,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2023/sample_orig_2023.txt
sample_orig_2024.txt,6.06,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2024/sample_orig_2024.txt
sample_orig_2025.txt,6.05,dbfs:/Volumes/workspace/default/freddie_mac_raw/DB Project/sample_2025/sample_orig_2025.txt


Total text files found: 20


In [0]:
from collections import Counter

expected_files = {
    f"sample_{kind}_{year}.txt"
    for year in range(2016, 2026)
    for kind in ["orig", "perf"]
}

filename_counts = Counter(f.name for f in project_files)

missing = sorted(expected_files - set(filename_counts))
unexpected = sorted(set(filename_counts) - expected_files)
duplicates = sorted(
    name for name, count in filename_counts.items() if count > 1
)
empty = sorted(f.name for f in project_files if f.size == 0)

print("Missing files:", missing)
print("Unexpected text files:", unexpected)
print("Duplicate filenames:", duplicates)
print("Empty files:", empty)

Missing files: []
Unexpected text files: []
Duplicate filenames: []
Empty files: []


In [0]:
# Folder containing this project's 2016–2025 data.
project_path = "/Volumes/workspace/default/freddie_mac_raw/DB Project/"

# Find text files inside this folder and all its subfolders.
def find_text_files(folder):
    results = []

    for item in dbutils.fs.ls(folder):
        if item.isDir():
            results.extend(find_text_files(item.path))
        elif item.name.endswith(".txt"):
            results.append(item)

    return results

# Create the variable needed by the validation cell.
project_files = find_text_files(project_path)

print(f"Text files found: {len(project_files)}")

Text files found: 20


In [0]:
from pyspark.sql import functions as F
from collections import Counter

project_path = "/Volumes/workspace/default/freddie_mac_raw/DB Project/"

# 1. Find the files inside all year subfolders.
def find_text_files(folder):
    results = []

    for item in dbutils.fs.ls(folder):
        if item.isDir():
            results.extend(find_text_files(item.path))
        elif item.name.endswith(".txt"):
            results.append(item)

    return results

project_files = find_text_files(project_path)
print(f"Text files found: {len(project_files)}")

# 2. Validate the file inventory.
expected_files = {
    f"sample_{kind}_{year}.txt"
    for year in range(2016, 2026)
    for kind in ["orig", "perf"]
}

filename_counts = Counter(f.name for f in project_files)

missing = sorted(expected_files - set(filename_counts))
unexpected = sorted(set(filename_counts) - expected_files)
duplicates = sorted(
    name for name, count in filename_counts.items() if count > 1
)
empty = sorted(f.name for f in project_files if f.size == 0)

print("Missing files:", missing)
print("Unexpected files:", unexpected)
print("Duplicate filenames:", duplicates)
print("Empty files:", empty)

assert not missing, f"Missing files: {missing}"
assert not unexpected, f"Unexpected files: {unexpected}"
assert not duplicates, f"Duplicate filenames: {duplicates}"
assert not empty, f"Empty files: {empty}"
assert len(project_files) == 20, "Expected exactly 20 files."

# 3. Read the files as raw text and check their field counts.
raw_lines = (
    spark.read.text([f.path for f in project_files])
    .select(
        F.col("value").alias("raw_line"),
        F.col("_metadata.file_name").alias("source_file"),
        F.col("_metadata.file_path").alias("source_path")
    )
)

structure_check = (
    raw_lines
    .withColumn(
        "field_count",
        F.size(F.split(F.col("raw_line"), r"\|", -1))
    )
    .groupBy("source_file", "field_count")
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.when(F.trim(F.col("raw_line")) == "", 1).otherwise(0)
        ).alias("blank_rows")
    )
    .orderBy("source_file", "field_count")
)

display(structure_check)

Text files found: 20
Missing files: []
Unexpected files: []
Duplicate filenames: []
Empty files: []
